# 2 · Finding a good route without trying them all

The shipped demonstration has **3,802,314,700,800** routes. Enumerating them is
not a plan, and "use a heuristic" is not one either.

The useful reframing is information-theoretic, and it makes the problem look
completely different:

> A route is one choice per sub-step. With no knowledge, naming a good one costs
> **Σ log₂(candidates) = 41.8 bits**. Every measured run supplies some of those
> bits. **The number of experiments scales with the entropy of your posterior,
> not with the size of the space.**

41.8 bits sounds hopeless until you notice it decomposes into 14 independent
choices of 2 to 6 bits each.

In [1]:
# Nothing here needs a browser, a model or a network. The core is stdlib-only.
#
# Installed from the repository rather than from a pinned release wheel: these
# notebooks use `types`, `facets` and `viz`, and pinning v0.3.0 meant installing
# a build from before those existed — so the notebook failed at cell one while
# looking, from the source, entirely correct.
try:
    import browsergraph  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "browsergraph @ git+https://github.com/"
                    "aidonerightcorp/browsergraph.git"], check=True)

import browsergraph as bg
print("browsergraph", bg.__version__)

browsergraph 0.3.0


In [2]:
from browsergraph.demo import workbench
from browsergraph.evidence import Evidence, stages_of
from browsergraph.policy import Policy

wb = workbench()
stages = stages_of(wb, Policy.permissive())
store = Evidence()

print(wb.summary())
print()
print(f"{wb.route_count():,} routes  =  {store.bits_of_choice(stages):.1f} bits of choice")
for sid, cs in list(stages.items())[:5]:
    import math
    print(f"   {sid:<12} {len(cs):>3} candidates = {math.log2(len(cs)):.1f} bits")

6 stages / 14 sub-steps · 57 definitions · 166 atomic candidates · 3,802,314,700,800 complete routes · 1,337 adjacent transitions

3,802,314,700,800 routes  =  41.8 bits of choice
   resolve        4 candidates = 2.0 bits
   session       70 candidates = 6.1 bits
   read           6 candidates = 2.6 bits
   decode         6 candidates = 2.6 bits
   parse         11 candidates = 3.5 bits


## Policy first, and it is a hard gate

Before anything is scored, candidates are filtered by what the task is
*permitted* to do. A candidate lacking a permission is **unavailable**, not
low-scoring — no amount of "but it scores well" may promote it. Get this
backwards and you build a system that confidently recommends something it is not
allowed to run.

In [3]:
from browsergraph.policy import review

locked = Policy(
    permissions=frozenset({"filesystem", "filesystem:read", "filesystem:write",
                           "database", "database:read"}),
    allow_external_effects=False, deterministic_only=True, name="locked-down")

report = review(wb, locked)
print(f"{wb.route_count():,} routes before policy")
print(f"{report.reachable_routes:,} after "
      f"({100 * (1 - report.reachable_routes / wb.route_count()):.2f}% removed)")
print()
for gate in list(report.gates.values())[:3]:
    print(" ", gate.summary())
    for verdict in gate.blocked[:2]:
        print(f"      {verdict.reason}")

3,802,314,700,800 routes before policy
1,959,552,000 after (99.95% removed)

  resolve: 4 of 4 candidates eligible
  session: 7 of 70 candidates eligible
      needs browser, network — not granted
      needs browser, network — not granted
  read: 3 of 6 candidates eligible
      needs browser — not granted
      needs browser — not granted


Every removal states its reason, and blocked candidates stay **visible**.
Filtering them out silently would answer "what could perform this step?" with
"what the policy left", and the screen would look identical either way.

## Three search strategies

**greedy** takes the best option per stage independently. **beam** keeps the most
promising partial routes. **exhaustive** enumerates — so "best" means best rather
than best-found.

Greedy is sometimes wrong, and the reason is worth understanding: route quality
**compounds**. A route is only as good as the joint probability that every step
worked, so a stage's real contribution depends on what the rest already spent.

In [4]:
from browsergraph import search

rows = []
for profile in wb.optimization_profiles:
    got = search.compare_strategies(wb, profile, policy=locked)
    rows.append((profile.name, got["greedy"], got["beam"]))

print(f"{'profile':<16}{'greedy':>9}{'beam':>9}   {'evaluations':>12}")
for name, g, b in rows:
    print(f"{name:<16}{g.score:>9.4f}{b.score:>9.4f}   {g.examined:>5} vs {b.examined:<5}")

profile            greedy     beam    evaluations
Balanced           1.0875   1.0875      71 vs 512  
Quality first      1.1277   1.1277      71 vs 512  
Speed first        0.9608   1.0304      71 vs 512  
Cost first         1.1478   1.1478      71 vs 512  


## Why it chose that — with numbers that add up

A route is an assertion until you can interrogate it. Each sub-step records how
many candidates were eligible, how many policy blocked, what won, its runner-up,
and **what each objective actually contributed** — the realised weighted terms,
not the profile's declared weights. A weight of 0.7 on a metric every candidate
shares contributes nothing, and only the realised term shows that.

In [5]:
proposal = search.propose(wb, wb.optimization_profiles[0],
                          policy=Policy.permissive(), strategy="greedy")
for d in proposal.decisions[:6]:
    terms = "  ".join(f"{m}={v:+.3f}" for m, v in sorted(d.contributions.items()))
    print(f"{d.stage:<12} score={d.score:.3f}  sum={sum(d.contributions.values()):.3f}  {terms}")
print()
print("the parts sum to the whole for every sub-step:",
      all(abs(sum(d.contributions.values()) - d.score) < 1e-9
          for d in proposal.decisions))

resolve      score=1.000  sum=1.000  latency_ms=+0.333  quality=+0.667
session      score=1.000  sum=1.000  latency_ms=+0.333  quality=+0.667
read         score=1.000  sum=1.000  latency_ms=+0.333  quality=+0.667
decode       score=1.000  sum=1.000  latency_ms=+0.333  quality=+0.667
parse        score=1.000  sum=1.000  cost_usd=+0.250  latency_ms=+0.250  quality=+0.500
normalize    score=0.966  sum=0.966  latency_ms=+0.299  quality=+0.667

the parts sum to the whole for every sub-step: True


## Learning: Thompson sampling at *sum* cost

Keep a posterior per candidate per context. Propose by sampling each sub-step —
**166 draws, not 3.8 trillion evaluations**. Exploration falls out of the
posterior's width; there is no ε to tune and it stops on its own.

The simulation below invents a hidden "true" success rate per candidate that the
learner cannot see, then watches it find them.

In [6]:
import random
from browsergraph.evidence import Observation, context_chain

rng = random.Random(7)
truth = {c: rng.betavariate(2, 3) for cs in stages.values() for c in cs}
best = {sid: max(cs, key=lambda c: truth[c]) for sid, cs in stages.items()}
ctx = context_chain("site:acme.com", "sector:retail")

store = Evidence()
print(f"{'runs':>6}{'bits left':>11}{'resolved':>10}{'correct picks':>15}")
for target in (0, 50, 200, 600):
    while sum(p.runs for c in store.posteriors.values()
              for p in c.values()) < target * len(stages):
        route = store.suggest(stages, ctx, seed=rng.randrange(1 << 30))
        for c in route.values():
            store.observe(Observation(candidate=c, context=ctx[0],
                                      ok=rng.random() < truth[c]))
    hits = sum(1 for sid, cs in stages.items()
               if store.ranked(cs, ctx)[0][0] == best[sid])
    print(f"{target:>6}{store.bits_remaining(stages, ctx):>11.1f}"
          f"{store.resolved(stages, ctx):>10.0%}{hits:>10} of {len(stages)}")

  runs  bits left  resolved  correct picks
     0       40.3        4%         0 of 14
    50       23.7       43%         7 of 14


   200       18.7       55%        11 of 14

   600        9.6       77%        14 of 14


## The finding that justifies per-step receipts

Same simulation, one change: learn only from whether the **whole run** passed,
instead of from each step's own outcome.

In [7]:
def learn(per_step, runs=600):
    store, local = Evidence(), random.Random(3)
    for _ in range(runs):
        route = store.suggest(stages, seed=local.randrange(1 << 30))
        q = 1.0
        for c in route.values():
            q *= truth[c]
        if per_step:
            for c in route.values():
                store.observe(Observation(candidate=c,
                                          ok=local.random() < truth[c]))
        else:
            store.observe_route(list(route.values()), quality=q,
                                ok=local.random() < q ** (1 / len(route)))
    hits = sum(1 for sid, cs in stages.items()
               if store.ranked(cs)[0][0] == best[sid])
    return store.resolved(stages), hits

for label, per_step in (("route-level pass/fail", False), ("per-step outcomes", True)):
    resolved, hits = learn(per_step)
    print(f"{label:<24} {resolved:>6.0%} resolved   {hits:>2} of {len(stages)} correct")

route-level pass/fail       27% resolved    5 of 14 correct


per-step outcomes           72% resolved   12 of 14 correct


Route-level success dilutes credit across fourteen candidates equally, so every
posterior converges to the *average* route quality rather than its own truth.
More runs do not help — the signal is not there.

That is a **measured** argument for something the design already had for other
reasons: per-step receipts and independent verification are not bookkeeping,
they are what makes learning tractable at all.

## Where independence breaks

Cheap search assumes the choices are independent. That assumption should be
measured, not believed: when a route does much worse than the product of its
parts predicted, *that specific pair* deserves joint search. Everything else can
stay greedy.

In [8]:
store = Evidence()
a, b = list(stages.values())[0][0], list(stages.values())[1][0]
for _ in range(12):                     # each is fine alone
    store.observe(Observation(candidate=a, ok=True))
    store.observe(Observation(candidate=b, ok=True))
for _ in range(6):                      # together they are not
    store.routes.append(((a, b), "global", 0.05))

for x, y, gap, runs in store.interactions():
    print(f"{x}\n  + {y}\n  {gap:+.2f} worse than independence predicted, over {runs} runs")

## The two regimes, and how they relate

- **Fast:** Thompson-sample a route, run it, fold the receipt back in. Suggested
  routes are the posterior's argmax; **fallbacks are its second and third place
  for this context**, not whatever was written down when the graph was drawn.
- **Exhaustive:** the same posteriors, swept rather than sampled. What that buys
  is not a better single answer — it is the interaction structure, the Pareto
  front, and the true second-best route.

They are the same machinery at different budgets. The cheap path is the
exhaustive path with a stopping rule, and `bits_remaining()` is the stopping
rule.